In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import joblib

print("🤖 Starting model training & comparison")

# =========================
# PATH CONFIGURATION (SAFE)
# =========================
try:
    BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
except NameError:
    BASE_DIR = os.getcwd()

DB_PATH = os.path.join(BASE_DIR, "data", "faculty.db")
MODEL_DIR = os.path.join(BASE_DIR, "models")

os.makedirs(MODEL_DIR, exist_ok=True)

TABLE_NAME = "faculty_workload"

# =========================
# LOAD DATA
# =========================
conn = sqlite3.connect(DB_PATH)
df = pd.read_sql(f"SELECT * FROM {TABLE_NAME}", conn)
conn.close()

print(f"📥 Data loaded: {df.shape}")

# =========================
# FEATURE / TARGET SPLIT
# =========================
X = df.drop(columns=["faculty_id", "workload_risk"])
y = df["workload_risk"]

# One-hot encode categorical features
X = pd.get_dummies(X, columns=["term", "exam_type"], drop_first=True)

# =========================
# TRAIN–TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# =========================
# FEATURE SCALING
# =========================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# =========================
# EVALUATION FUNCTION
# =========================
def evaluate_model(name, y_true, y_pred):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1 Score": f1_score(y_true, y_pred)
    }

results = []

# =========================
# 1️⃣ LOGISTIC REGRESSION
# =========================
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_scaled, y_train)

y_pred_log = log_model.predict(X_test_scaled)
results.append(evaluate_model("Logistic Regression", y_test, y_pred_log))

# =========================
# 2️⃣ LINEAR REGRESSION (BASELINE)
# =========================
lin_model = LinearRegression()
lin_model.fit(X_train_scaled, y_train)

# Convert regression output → binary
y_pred_lin = (lin_model.predict(X_test_scaled) >= 0.5).astype(int)
results.append(evaluate_model("Linear Regression", y_test, y_pred_lin))

# =========================
# MODEL COMPARISON
# =========================
results_df = pd.DataFrame(results)
print("\n📊 Model Comparison Results")
print(results_df)

# =========================
# SAVE FINAL MODEL
# =========================
joblib.dump(log_model, os.path.join(MODEL_DIR, "logistic_regression_model.joblib"))
joblib.dump(scaler, os.path.join(MODEL_DIR, "scaler.joblib"))

print("\n💾 Logistic Regression model saved")
print("🎯 Model training & comparison completed")

🤖 Starting model training & comparison


DatabaseError: Execution failed on sql 'SELECT * FROM faculty_workload_cleaned': no such table: faculty_workload_cleaned